In [81]:
# Імпортуємо необхідні бібліотеки
import pandas as pd
import numpy as np

import warnings
warnings.simplefilter("ignore")

In [82]:
df = pd.read_csv("customer_support_tickets.csv")
df.head(3)

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0


In [83]:
print(df.isna().sum())

Ticket ID                          0
Customer Name                      0
Customer Email                     0
Customer Age                       0
Customer Gender                    0
Product Purchased                  0
Date of Purchase                   0
Ticket Type                        0
Ticket Subject                     0
Ticket Description                 0
Ticket Status                      0
Resolution                      5700
Ticket Priority                    0
Ticket Channel                     0
First Response Time             2819
Time to Resolution              5700
Customer Satisfaction Rating    5700
dtype: int64


In [84]:
#Треба видалити зайві колонки, які не впливають на навчання моделі. Для прикладу ім'я не може вплинути на точність, а тільки погіршить роботу моделі.
df.drop(columns=['Customer Email', 'Customer Name', 'Ticket ID', 'Date of Purchase', 'Ticket Status'], inplace=True, errors='ignore')

#Так як наш цільовий клас це колонка Satisfaction то потрібно видалити всі рядки там де NaN
print(df.shape) #Кількість рядків до видалення рядків NaN

df = df.dropna(subset=['Customer Satisfaction Rating'])
print(df.shape) #Кількість рядків після видалення рядків NaN


(8469, 12)
(2769, 12)


In [85]:
#Як бачимо ми втратили велику кількість даних, але нічого іншого тут вже зробити не можна, якщо нам потрібен цільовий клас, а так могли б використати класифікацію.

In [86]:
df.info()

<class 'pandas.DataFrame'>
Index: 2769 entries, 2 to 8467
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Customer Age                  2769 non-null   int64  
 1   Customer Gender               2769 non-null   str    
 2   Product Purchased             2769 non-null   str    
 3   Ticket Type                   2769 non-null   str    
 4   Ticket Subject                2769 non-null   str    
 5   Ticket Description            2769 non-null   str    
 6   Resolution                    2769 non-null   str    
 7   Ticket Priority               2769 non-null   str    
 8   Ticket Channel                2769 non-null   str    
 9   First Response Time           2769 non-null   str    
 10  Time to Resolution            2769 non-null   str    
 11  Customer Satisfaction Rating  2769 non-null   float64
dtypes: float64(1), int64(1), str(10)
memory usage: 281.2 KB


In [87]:
from sklearn.model_selection import train_test_split

TARGET_NAME = 'Customer Satisfaction Rating'
X = df.drop(columns=[TARGET_NAME])
y = df[TARGET_NAME]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

In [88]:
#Треба тепер попрацювати для даних з часом

def create_time_features(X):
    X = X.copy()

    date_cols = ['First Response Time', 'Time to Resolution']

    for col in date_cols:
        X[col] = pd.to_datetime(X[col], errors='coerce')


    X['Response to Resolution'] = (
        X['Time to Resolution'] - X['First Response Time']
    ).dt.total_seconds() / 3600

    return X

X_train = create_time_features(X_train)
X_valid = create_time_features(X_valid)

In [89]:
#Зверху ми створили функцію яка рахує кількість годин від фідбека до вирішення проблеми тому можемо видалити зайві колонки з datedtime
X_train.drop(columns=['Time to Resolution', 'First Response Time'], inplace=True, errors='ignore')
X_valid.drop(columns=['Time to Resolution', 'First Response Time'], inplace=True, errors='ignore')

#Видаляємо зайві слова у Ticket Description
X_train['Ticket Description'] = X_train['Ticket Description'].str.replace('{product_purchased}', '', regex=False)
X_valid['Ticket Description'] = X_valid['Ticket Description'].str.replace('{product_purchased}', '', regex=False)

print(X_train.head(3))

      Customer Age Customer Gender Product Purchased      Ticket Type  \
7841            66           Other         Canon EOS  Technical issue   
5118            22           Other       Amazon Echo  Product inquiry   
3439            33            Male       HP Pavilion  Product inquiry   

            Ticket Subject                                 Ticket Description  \
7841  Installation support  I'm having an issue with the . Please assist. ...   
5118         Display issue  The  is unable to establish a stable internet ...   
3439         Product setup  I'm having an issue with the . Please assist.\...   

                                             Resolution Ticket Priority  \
7841  I relationship name always western knowledge m...          Medium   
5118                   Congress else research customer.          Medium   
3439                    Early front evidence view also.        Critical   

     Ticket Channel  Response to Resolution  
7841          Email               -

In [90]:
number_cols = X_train.select_dtypes(include='number').columns.tolist()

exclude_cols =  ['Product Purchased', 'Ticket Description', 'Resolution'] #Колонки які не є категоріями потрібно виключити
cat_cols = [
    col for col in X_train.select_dtypes(include='object').columns if col not in exclude_cols
]

In [91]:
# Робимо OHE для категоріальних ознак.
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_ohe = ohe.fit_transform(X_train[cat_cols])
X_valid_ohe = ohe.transform(X_valid[cat_cols])

In [92]:
X_train_ohe_df = pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out(cat_cols), index=X_train.index)
X_valid_ohe_df = pd.DataFrame(X_valid_ohe, columns=ohe.get_feature_names_out(cat_cols), index=X_valid.index)

In [93]:
X_train_ohe_df.head(4)

,Customer Gender_Female,Customer Gender_Male,Customer Gender_Other,Ticket Type_Billing inquiry,Ticket Type_Cancellation request,Ticket Type_Product inquiry,Ticket Type_Refund request,Ticket Type_Technical issue,Ticket Subject_Account access,Ticket Subject_Battery life,...,Ticket Subject_Refund request,Ticket Subject_Software bug,Ticket Priority_Critical,Ticket Priority_High,Ticket Priority_Low,Ticket Priority_Medium,Ticket Channel_Chat,Ticket Channel_Email,Ticket Channel_Phone,Ticket Channel_Social media
7841,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
5118,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3439,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2003,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [117]:
# Обробляємо колонками з number значеннями.
# Будемо робити стандартизацію (або z-score normalization).
# Її суть у тому, щоб привести сирі дані, які можуть містити величезний розброс -
# до середнього значення 0 і стандартного відхилення - 1.

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[number_cols])
X_valid_scaled = scaler.transform(X_valid[number_cols])

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=number_cols, index=X_train.index)
X_valid_scaled_df = pd.DataFrame(X_valid_scaled, columns=number_cols, index=X_valid.index)

In [95]:
X_train_scaled_df.head(3)

,Customer Age,Response to Resolution
7841,1.434036,-0.081398
5118,-1.467318,2.424170
3439,-0.741980,0.203405


In [96]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_desc = TfidfVectorizer(
    max_features=500,
    stop_words='english'
)

def VectorizeColumnTest(X, column_name):
    return tfidf_desc.fit_transform(X[column_name].fillna(''))

def VectorizeColumnValid(X, column_name):
    return tfidf_desc.transform(X[column_name].fillna(''))

X_train_desc = VectorizeColumnTest(X_train, 'Ticket Description')
X_valid_desc = VectorizeColumnValid(X_valid, 'Ticket Description')

X_train_res = VectorizeColumnTest(X_train, 'Resolution')
X_valid_res = VectorizeColumnValid(X_valid, 'Resolution')

X_train_prod = VectorizeColumnTest(X_train, 'Product Purchased')
X_valid_prod = VectorizeColumnValid(X_valid, 'Product Purchased')

In [97]:
X_train_final_df = pd.concat([X_train_scaled_df, X_train_ohe_df], axis=1)
X_valid_final_df = pd.concat([X_valid_scaled_df, X_valid_ohe_df], axis=1)

#І тепер фінальне об'єднання з векторизованими даними
from scipy.sparse import hstack, csr_matrix

X_train_final = hstack([
    csr_matrix(X_train_final_df.to_numpy()),
    X_train_desc,
    X_train_res,
    X_train_prod
]).tocsr()

X_valid_final = hstack([
    csr_matrix(X_valid_final_df.to_numpy()),
    X_valid_desc,
    X_valid_res,
    X_valid_prod
]).tocsr()

In [105]:
# Перевіримо, що ніякі дані не втратились протягом наших маніпуляций.
print(X_train.shape[0] == X_train_final.shape[0])
# Наостанок перевіримо чи є пропуски в даних перед навчанням моделі.
print(np.isnan(X_train_final.data).sum())
print()
print(X_train_final.shape)
print(X_valid_final.shape)

True
0

(2215, 1105)
(554, 1105)


In [109]:
from sklearn.linear_model import LinearRegression

# Спочатку побудуємо модель лінійної регресії.
lin_model = LinearRegression()
lin_model.fit(X_train_final, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1105,)","[-0. , 0.01, 0.03,..., 0.14,-0.81,-0.3 ]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,3.787
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1105


In [110]:
# Зробимо передбачення спочатку на тренувальних даних.
y_train_pred = lin_model.predict(X_train_final)

In [111]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MAE:", mean_absolute_error(y_train, y_train_pred))
print("MSE:", mean_squared_error(y_train, y_train_pred))
print("R²:", r2_score(y_train, y_train_pred))

MAE: 0.8361098935948841
MSE: 1.0446915089260354
R²: 0.4725252022961902


In [112]:
#На тренувальних даних даних модель ідеально себе показала, зараз перевіримо валідаційні дані

y_valid_pred = lin_model.predict(X_valid_final)
print("MAE:", mean_absolute_error(y_valid, y_valid_pred))
print("MSE:", mean_squared_error(y_valid, y_valid_pred))
print("R²:", r2_score(y_valid, y_valid_pred))


MAE: 1.49849571309749
MSE: 3.3704273266049
R²: -0.708682465625708


In [114]:
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor(strategy='mean')

dummy.fit(X_train_final, y_train)

y_dummy_pred = dummy.predict(X_valid_final)

print("Dummy MAE:", mean_absolute_error(y_valid, y_dummy_pred))
print("Dummy R²:", r2_score(y_valid, y_dummy_pred))

Dummy MAE: 1.19061127364295
Dummy R²: -9.91735386777215e-05


In [115]:
from sklearn.linear_model import Ridge

ridge_model = Ridge(alpha=10.0)

ridge_model.fit(X_train_final, y_train)

y_train_ridge = ridge_model.predict(X_train_final)
y_valid_ridge = ridge_model.predict(X_valid_final)

print("Train MAE:", mean_absolute_error(y_train, y_train_ridge))
print("Valid MAE:", mean_absolute_error(y_valid, y_valid_ridge))
print("Valid R²:", r2_score(y_valid, y_valid_ridge))

Train MAE: 1.084190851451895
Valid MAE: 1.24147491435452
Valid R²: -0.05284836531610293


In [ ]:
#Висновок: модель не змогла знайти закономірності в задоволеності клієнтів.

#БІЛЬШ ДЕТАЛЬНО ЧИТАЙТЕ В README!